<a href="https://colab.research.google.com/github/joryhh/Capstone-project-agentic-AI-systems-engineering/blob/main/phase0_state_and_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# STEP 1 — SHARED STATE SCHEMA
# ============================================================

from typing import TypedDict, List, Dict, Optional, Literal, Annotated
import operator

# ---- Sub-schemas: the actual inter-agent "message contract" ----

class SearchResult(TypedDict):
    source: str            # "flight_search" | "hotel_search" | "attraction_search"
    query: str
    raw_content: str
    url: Optional[str]

class ReActStep(TypedDict):
    thought: str
    action: str
    action_input: str
    observation: str

class ItineraryItem(TypedDict):
    day: int
    activity: str
    location: str
    estimated_cost: float
    category: Literal["flight", "accommodation", "transit", "activity", "food", "entry_fee"]

class BudgetBreakdown(TypedDict):
    total_estimated_cost: float
    budget_limit: float
    over_budget: bool
    over_budget_amount: float
    cost_by_category: Dict[str, float]
    constraint_for_replanning: Optional[str]  # <- the concrete constraint B sends back to Agent 1

class GuardrailLog(TypedDict):
    check_type: Literal["prompt_injection", "pii_masking"]
    triggered: bool
    details: str
    timestamp: str

# ---- The shared State object ----

class TravelState(TypedDict):
    # Input (set once, at invoke time)
    user_request: str
    destination: str
    travel_dates: str
    budget_limit: float
    traveler_preferences: List[str]

    # Agent 1 (Planner / ReAct)
    react_trace: Annotated[List[ReActStep], operator.add]
    search_results: Annotated[List[SearchResult], operator.add]
    draft_itinerary: List[ItineraryItem]

    # Agent 2 (Budget)
    budget_analysis: Optional[BudgetBreakdown]

    # Agent 3 (Audit)
    final_summary: Optional[str]
    audit_notes: List[str]

    # Guardrails
    guardrail_logs: Annotated[List[GuardrailLog], operator.add]
    pii_masked: bool

    # Control flow
    iteration_count: int
    max_iterations: int
    replan_reason: Optional[str]

    # HITL
    human_approved: Optional[bool]
    human_feedback: Optional[str]

    # Cross-cutting
    execution_logs: Annotated[List[str], operator.add]
    status: Literal["in_progress", "awaiting_human", "completed", "failed"]

In [4]:
# CORRECT — partial return, reducer appends automatically
def agent1_planner_node(state: TravelState) -> dict:
    return {
        "execution_logs": ["Agent 1 (Planner) stub executed"],
        "iteration_count": state["iteration_count"] + 1,
    }




In [5]:
# ============================================================
# STEP 4 — GRAPH SKELETON (stub nodes, real edges + real conditional logic)
# ============================================================

from langgraph.graph import StateGraph, END
from typing import Literal

# ---- Stub nodes  ----

def agent1_planner_node(state: TravelState) -> dict:
    print("  [Agent 1 - Planner] STUB running.")
    return {
        "execution_logs": ["Agent 1 (Planner) stub executed"],
        "iteration_count": state["iteration_count"] + 1,
    }

def agent2_budget_node(state: TravelState) -> dict:
    print("  [Agent 2 - Budget] STUB running.")
    return {
        "execution_logs": ["Agent 2 (Budget) stub executed"],
    }

def agent3_audit_node(state: TravelState) -> dict:
    print("  [Agent 3 - Audit] STUB running.")
    return {
        "execution_logs": ["Agent 3 (Audit) stub executed"],
        "status": "completed",
    }

def human_review_node(state: TravelState) -> dict:
    print("  [Human Review] STUB — B will build this as the real HITL interrupt.")
    return {
        "execution_logs": ["Human review stub executed"],
    }

# ---- The router: your termination guard + conditional edge ----

def budget_router(state: TravelState) -> Literal["replan", "proceed"]:
    # Termination guard FIRST — this must win regardless of budget status,
    # or an unfixable budget constraint loops forever.
    if state["iteration_count"] >= state["max_iterations"]:
        print(f"  [Router] Max iterations ({state['max_iterations']}) reached -> forcing proceed.")
        return "proceed"

    budget = state.get("budget_analysis")
    if budget and budget["over_budget"]:
        print(f"  [Router] Over budget by {budget['over_budget_amount']} -> replanning.")
        return "replan"

    print("  [Router] Within budget -> proceeding.")
    return "proceed"

# ---- Assemble the graph ----

workflow = StateGraph(TravelState)

workflow.add_node("agent1_planner", agent1_planner_node)
workflow.add_node("agent2_budget", agent2_budget_node)
workflow.add_node("agent3_audit", agent3_audit_node)
workflow.add_node("human_review", human_review_node)

workflow.set_entry_point("agent1_planner")
workflow.add_edge("agent1_planner", "agent2_budget")

workflow.add_conditional_edges(
    "agent2_budget",
    budget_router,
    {
        "replan": "agent1_planner",   # loop back
        "proceed": "agent3_audit",    # move forward
    },
)

workflow.add_edge("agent3_audit", "human_review")
workflow.add_edge("human_review", END)

app = workflow.compile()
print("Graph compiled and ready.")

Graph compiled and ready.


In [6]:
# ============================================================
# STEP 5 — SMOKE TEST: run the skeleton, confirm the loop + guard work
# ============================================================

initial_state: TravelState = {
    "user_request": "5-day trip to Kyoto, mid-range budget",
    "destination": "Kyoto",
    "travel_dates": "2026-11-10 to 2026-11-15",
    "budget_limit": 2000.0,
    "traveler_preferences": ["food", "temples"],
    "react_trace": [],
    "search_results": [],
    "draft_itinerary": [],
    "budget_analysis": None,
    "final_summary": None,
    "audit_notes": [],
    "guardrail_logs": [],
    "pii_masked": False,
    "iteration_count": 0,
    "max_iterations": 3,
    "replan_reason": None,
    "human_approved": None,
    "human_feedback": None,
    "execution_logs": [],
    "status": "in_progress",
}

# Belt-and-suspenders: LangGraph's own recursion_limit as a second safety net,
# independent of your max_iterations field — good practice for any graph with a cycle.
final_state = app.invoke(initial_state, config={"recursion_limit": 50})

print("\nFinal status:", final_state["status"])
print("Iterations used:", final_state["iteration_count"])
for log in final_state["execution_logs"]:
    print(log)

  [Agent 1 - Planner] STUB running.
  [Agent 2 - Budget] STUB running.
  [Router] Within budget -> proceeding.
  [Agent 3 - Audit] STUB running.
  [Human Review] STUB — B will build this as the real HITL interrupt.

Final status: completed
Iterations used: 1
Agent 1 (Planner) stub executed
Agent 2 (Budget) stub executed
Agent 3 (Audit) stub executed
Human review stub executed
